# Aim 2 + Aim 3 Colab Pro Pipeline: Multi-Causal `SuSiE` + `coloc.susie` + `HyPrColoc` + `1000G ANX SMR-multi` + Cell-Type Brain eQTL

**Why run Aim 2 + Aim 3 in Google Colab Pro?**
- Google Colab runs on **Ubuntu Linux x86_64**, so precompiled Linux binaries (**`smr-1.3.1-linux-x86_64`**, **`plink1.9`**) and CRAN multi-causal R packages (**`coloc`**, **`susieR`**, **`hyprcoloc`**, **`ctwas`**) install cleanly in ~2–3 minutes.
- Directly mounts your **5 TB Google Drive** (`MyDrive`), eliminating the 50 GB local disk bottleneck on Mac while downloading large public panels (**1000G EUR LD**, **Bryois 2022 8 Brain Cell-Type eQTL**, **BrainMeta BESD**) at datacenter speed.
- **Compute Units Saver**: Select **Runtime -> Change runtime type -> CPU + High-RAM** (or standard RAM; Aim 2/3 run per-locus windows using ~2–16 GB RAM and **do NOT need A100 GPU**, saving almost all compute units for Aim 1!).

## Files to Upload Once to Your Google Drive (`MyDrive/MDD_New_Project_Pipelines_2026/03_aim2_inputs/`)
Upload from your Mac:
1. `03_Aim2_Aim3_Server_Scripts/outputs/loci_dualtrait/*.tsv` (7 extracted dual-trait N=1.15M MDD + N=1.10M ANX windows, < 5 MB total!)
2. `MDD_no23andMe_EUR_SMR.ma.gz` (130 MB) & `ANX_EUR_GWAShug_aligned3.tsv.gz` (134 MB) from `02_Scheme2_GWAS_Causal_Coloc_SMR/inputs/large_n_gwas_archive/`
3. Optional: If downloaded manually from Yang Lab Portal, place `BrainMeta_cis_eQTL_chr*.besd` / `cis_sQTL` BESD bundles into `MyDrive/MDD_New_Project_Pipelines_2026/03_drive_cache/brainmeta_besd/`

In [ ]:
# [1] Mount 5 TB Google Drive & Inspect Available System RAM + Disk Space
import os, shutil, subprocess, json
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/MDD_New_Project_Pipelines_2026'
DRIVE_IN    = os.path.join(DRIVE_ROOT, '03_aim2_inputs')
DRIVE_CACHE = os.path.join(DRIVE_ROOT, '03_drive_cache')
DRIVE_OUT   = os.path.join(DRIVE_ROOT, '03_aim2_aim3_outputs')
WORK_DIR    = '/content/aim2_work'

for d in [
    DRIVE_IN, DRIVE_OUT,
    os.path.join(DRIVE_CACHE, 'ld_1kg_eur'),
    os.path.join(DRIVE_CACHE, 'bryois_zenodo_5543734'),
    os.path.join(DRIVE_CACHE, 'brainmeta_besd'),
    os.path.join(DRIVE_CACHE, 'bins'),
    os.path.join(WORK_DIR, 'ld_matrices'),
    os.path.join(WORK_DIR, 'smr_multi'),
    os.path.join(WORK_DIR, 'results'),
]:
    os.makedirs(d, exist_ok=True)

subprocess.run(['df', '-h', '/content', '/content/drive/MyDrive'], check=True)
subprocess.run(['free', '-h'], check=True)

In [ ]:
# [2] One-Click Install Linux Bio Tools (PLINK 1.9, SMR 1.4.3, R: coloc, susieR, hyprcoloc)
import subprocess, os, zipfile, re, urllib.request, ssl, shutil, sys

def run_live(cmd_list, desc=''):
    if desc:
        print(desc, flush=True)
    p = subprocess.Popen(cmd_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd_list)

print('[Step 1/3] Updating apt and installing Linux dependencies & PLINK 1.9...', flush=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'plink1.9', 'unzip', 'wget', 'git', 'cmake', 'g++', 'libopenblas-dev', 'libgsl-dev', 'r-base', 'r-cran-data.table', 'r-cran-remotes', 'r-cran-matrix', 'r-cran-rcpp'], check=True)
if not os.path.exists('/usr/local/bin/plink') and os.path.exists('/usr/bin/plink1.9'):
    os.symlink('/usr/bin/plink1.9', '/usr/local/bin/plink')

print('\n[Step 2/3] Setting up Yang Lab official SMR Linux x86_64 binary...', flush=True)
smr_cached_bin = os.path.join(DRIVE_CACHE, 'bins', 'smr_linux_x86_64')
smr_zip = os.path.join(DRIVE_CACHE, 'bins', 'smr_linux.zip')

if not os.path.exists(smr_cached_bin):
    cand_urls = [
        'https://yanglab.westlake.edu.cn/software/smr/download/smr-1.4.3-linux-x86_64.zip',
        'https://yanglab.westlake.edu.cn/software/smr/download/smr-1.3.1-linux-x86_64.zip',
    ]
    ssl_ctx = ssl.create_default_context()
    ssl_ctx.check_hostname = False
    ssl_ctx.verify_mode = ssl.CERT_NONE
    ua_headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

    try:
        req = urllib.request.Request('https://yanglab.westlake.edu.cn/software/smr/', headers=ua_headers)
        with urllib.request.urlopen(req, context=ssl_ctx, timeout=15) as resp:
            html = resp.read().decode('utf-8', errors='replace')
        found = re.findall(r'href=["\']([^"\']*(?:smr|download)[^"\']*linux[^"\']*\.zip)["\']', html, re.IGNORECASE)
        for u in reversed(found):
            full_u = u if u.startswith('http') else ('https://yanglab.westlake.edu.cn/software/smr/' + u.lstrip('./'))
            if full_u not in cand_urls:
                cand_urls.insert(0, full_u)
    except Exception as ex:
        print('  Note: SMR portal discovery skipped:', ex, flush=True)

    for url in cand_urls:
        if os.path.exists(smr_zip) and not zipfile.is_zipfile(smr_zip):
            os.remove(smr_zip)
        if not os.path.exists(smr_zip):
            print('  Trying download:', url, flush=True)
            try:
                dl_req = urllib.request.Request(url, headers=ua_headers)
                with urllib.request.urlopen(dl_req, context=ssl_ctx, timeout=25) as r, open(smr_zip, 'wb') as f_out:
                    shutil.copyfileobj(r, f_out)
            except Exception as dl_ex:
                print('  Download error with urlopen, trying wget:', dl_ex, flush=True)
                subprocess.run(['wget', '-q', '--timeout=20', '--user-agent="Mozilla/5.0"', url, '-O', smr_zip], check=False)
        if os.path.exists(smr_zip) and zipfile.is_zipfile(smr_zip):
            print('  -> Verified valid ZIP archive!', flush=True)
            break

    if os.path.exists(smr_zip) and zipfile.is_zipfile(smr_zip):
        subprocess.run(['unzip', '-q', '-o', smr_zip, '-d', '/content/smr_bin'], check=True)
        for root_d, _, files in os.walk('/content/smr_bin'):
            for fn in files:
                if fn.startswith('smr') and not fn.endswith('.zip'):
                    cand = os.path.join(root_d, fn)
                    shutil.copy2(cand, smr_cached_bin)
                    break
    else:
        print('  Binary ZIP unavailable; building SMR natively from GitHub source (~1-2 min)...', flush=True)
        if os.path.exists(smr_zip):
            os.remove(smr_zip)
        subprocess.run(['rm', '-rf', '/content/SMR_src'], check=True)
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/JianYang-Lab/SMR.git', '/content/SMR_src'], check=True)
        os.makedirs('/content/SMR_src/build', exist_ok=True)
        subprocess.run(['cmake', '..'], cwd='/content/SMR_src/build', check=True)
        subprocess.run(['make', '-j4'], cwd='/content/SMR_src/build', check=True)
        for cand_name in ['smr', 'smr-1.4.3', 'smr-1.3.1']:
            p_bin = os.path.join('/content/SMR_src/build', cand_name)
            if os.path.exists(p_bin):
                shutil.copy2(p_bin, smr_cached_bin)
                break

shutil.copy2(smr_cached_bin, '/usr/local/bin/smr')
os.chmod('/usr/local/bin/smr', 0o755)
print('  -> SMR successfully installed to /usr/local/bin/smr', flush=True)

print('\n[Step 3/3] Installing R multi-causal fine-mapping packages (coloc, susieR, hyprcoloc)...', flush=True)
print('  -> Live installation output will stream below line by line:', flush=True)

r_install_code = '''
options(Ncpus = 4L)
# Aim 2 fine-mapping core packages
for (p in c('coloc', 'susieR')) {
  if (!requireNamespace(p, quietly = TRUE)) {
    cat(sprintf('--> [R] Installing CRAN package: %s ...\n', p))
    install.packages(p, repos = 'https://cloud.r-project.org')
  } else {
    cat(sprintf('--> [R] Already installed: %s\n', p))
  }
}
if (!requireNamespace('hyprcoloc', quietly = TRUE)) {
  cat('--> [R] Installing GitHub package: jrs95/hyprcoloc ...\n')
  remotes::install_github('jrs95/hyprcoloc', upgrade = 'never')
} else {
  cat('--> [R] Already installed: hyprcoloc\n')
}
cat('\n============================================================\n')
cat('Aim 2 R packages ready: ', paste(sapply(c('data.table','coloc','susieR','hyprcoloc'), requireNamespace, quietly=TRUE), collapse=', '), '\n')
cat('============================================================\n')
'''

run_live(['Rscript', '-e', r_install_code])

print('\nVerifying CLI tools:', flush=True)
subprocess.run(['plink', '--version'], check=True)
subprocess.run(['smr', '--version'], check=False)
print('\n[SUCCESS] Environment setup complete! Ready to run Cell [3] & Cell [4].', flush=True)


In [ ]:
# [3] Download 1000G EUR PLINK Reference (488 MB, ~15s) & Stage Local NVMe Workspace
import os, glob, subprocess, shutil, zipfile

ld_dir = os.path.join(DRIVE_CACHE, 'ld_1kg_eur')
g1k_zip = os.path.join(ld_dir, 'g1000_eur.zip')
os.makedirs(ld_dir, exist_ok=True)

print('[Step 3A] Checking 1000 Genomes Phase 3 EUR reference panel...', flush=True)
if not os.path.exists(os.path.join(ld_dir, 'g1000_eur.bed')):
    if os.path.exists(g1k_zip):
        if not zipfile.is_zipfile(g1k_zip) or os.path.getsize(g1k_zip) < 100 * 1024 * 1024:
            print('  -> Removing incomplete/corrupted zip file...', flush=True)
            os.remove(g1k_zip)
    if not os.path.exists(g1k_zip):
        print('  -> Downloading 1000G EUR PLINK panel (488 MB) from official SURFdrive direct link...', flush=True)
        surfdrive_url = 'https://vu.data.surfsara.nl/index.php/s/VZNByNwpD8qqINe/download'
        subprocess.run(['wget', '-c', '--user-agent="Mozilla/5.0"', surfdrive_url, '-O', g1k_zip], check=True)
    print('  -> Unpacking 1000G EUR reference dataset...', flush=True)
    subprocess.run(['unzip', '-o', g1k_zip, '-d', ld_dir], check=True)

# Copy 1000G EUR bfiles to local Colab NVMe for fast PLINK/SMR execution
print('[Step 3B] Staging 1000G EUR bfiles to high-speed NVMe...', flush=True)
for ext in ['bed', 'bim', 'fam']:
    src_f = os.path.join(ld_dir, f'g1000_eur.{ext}')
    if os.path.exists(src_f):
        shutil.copy2(src_f, os.path.join(WORK_DIR, f'g1000_eur.{ext}'))
print('  -> 1000G EUR NVMe status: ready =', os.path.exists(os.path.join(WORK_DIR, 'g1000_eur.bed')), flush=True)

# Stage dual-trait inputs from Google Drive to local NVMe
print('\n[Step 3C] Checking user-uploaded dual-trait GWAS files in Drive...', flush=True)
loci_nvme = os.path.join(WORK_DIR, 'loci_dualtrait')
os.makedirs(loci_nvme, exist_ok=True)
staged_files = sorted(glob.glob(os.path.join(DRIVE_IN, '*.tsv')) + glob.glob(os.path.join(DRIVE_IN, '*.tsv.gz')) + glob.glob(os.path.join(DRIVE_IN, '*.ma')) + glob.glob(os.path.join(DRIVE_IN, '*.ma.gz')))
for p in staged_files:
    shutil.copy2(p, os.path.join(loci_nvme, os.path.basename(p)))

locus_tsvs = sorted(glob.glob(os.path.join(loci_nvme, '*_dualtrait_grch37.tsv')))
print(f'  -> Staged {len(staged_files)} total input files ({len(locus_tsvs)} dual-trait locus windows) on NVMe.', flush=True)
if len(locus_tsvs) == 0:
    print(f'  [!] Reminder: Please copy your 7 locus TSV files generated by 00_prepare_local_aim2_aligned_loci.py into:')
    print(f'      {DRIVE_IN}')
else:
    for i, lf in enumerate(locus_tsvs, 1):
        print(f'      ({i}) {os.path.basename(lf)}')
print('\n[SUCCESS] Cell [3] finished in seconds! You can now run Cell [4].', flush=True)


In [ ]:
# [4] Extract Per-Locus 1000G EUR LD Matrices & Run Multi-Causal SuSiE + coloc.susie (Live Progress)
import glob, os, subprocess, shutil, pandas as pd, sys

loci_nvme = os.path.join(WORK_DIR, 'loci_dualtrait')
os.makedirs(loci_nvme, exist_ok=True)

# Auto-detect where user uploaded files (whether in Drive, /content root, or 03_aim2_inputs)
existing_loci = glob.glob(os.path.join(loci_nvme, '*_dualtrait_grch37.tsv'))
if len(existing_loci) == 0:
    print('[Auto-Detect] Searching Colab and Drive for your uploaded locus files...', flush=True)
    found_count = 0
    for root, dirs, files in os.walk('/content'):
        if root == loci_nvme:
            continue
        for f in files:
            if f.endswith('_dualtrait_grch37.tsv') or f.endswith('_SMR.ma.gz') or f.endswith('aligned3.tsv.gz'):
                shutil.copy2(os.path.join(root, f), os.path.join(loci_nvme, f))
                found_count += 1
    if found_count > 0:
        print(f'  -> Successfully found and staged {found_count} uploaded files into workspace!', flush=True)

loci_files = sorted(glob.glob(os.path.join(loci_nvme, '*_dualtrait_grch37.tsv')))
print(f'[Step 4A] Found {len(loci_files)} dual-trait locus windows to fine-map.', flush=True)

print('\n[Step 4B] Computing 1000G EUR LD Pearson correlation matrices for each locus...', flush=True)
for i, lf in enumerate(loci_files, 1):
    tag = os.path.basename(lf).replace('_dualtrait_grch37.tsv', '')
    df = pd.read_csv(lf, sep='\t')
    chr_i = int(df['CHR'].iloc[0])
    bp_min, bp_max = int(df['BP'].min()), int(df['BP'].max())
    snp_list = os.path.join(WORK_DIR, 'ld_matrices', f'{tag}.snps')
    df[['SNP']].drop_duplicates().to_csv(snp_list, sep='\t', header=False, index=False)
    out_pref = os.path.join(WORK_DIR, 'ld_matrices', tag)
    print(f'  [{i}/{len(loci_files)}] Locus {tag} (chr{chr_i}:{bp_min:,}-{bp_max:,}, SNPs: {len(df):,}) -> PLINK...', end='', flush=True)
    subprocess.run([
        'plink', '--bfile', os.path.join(WORK_DIR, 'g1000_eur'),
        '--chr', str(chr_i), '--from-bp', str(bp_min), '--to-bp', str(bp_max),
        '--extract', snp_list, '--make-bed', '--out', out_pref
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    if os.path.exists(out_pref + '.bed'):
        # Generate uncompressed .ld directly (or gunzip) so R fread loads it instantly without requiring R.utils
        subprocess.run(['plink', '--bfile', out_pref, '--r', 'square', '--out', out_pref], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        print(' [LD OK]', flush=True)
    else:
        print(' [Warning: no matching SNPs in 1000G]', flush=True)

# Ensure any previous .ld.gz are uncompressed to plain .ld for zero-dependency R reading
for gz in glob.glob(os.path.join(WORK_DIR, 'ld_matrices', '*.ld.gz')):
    subprocess.run(['gunzip', '-f', gz], check=False)

print('\n[Step 4C] Running R SuSiE multi-causal fine-mapping & coloc.susie (Live Streaming)...', flush=True)
r_aim2 = fr'''
suppressPackageStartupMessages({{ library(data.table); library(coloc); library(susieR) }})
loci_files <- list.files('{WORK_DIR}/loci_dualtrait', pattern='_dualtrait_grch37\\.tsv$', full.names=TRUE)
cat(sprintf('--> Starting SuSiE fine-mapping across %d loci...\n', length(loci_files)))
res_list <- list()
for (i in seq_along(loci_files)) {{
  lf <- loci_files[i]
  tag <- sub('_dualtrait_grch37\\.tsv$', '', basename(lf))
  ld_f  <- file.path('{WORK_DIR}/ld_matrices', paste0(tag, '.ld'))
  if (!file.exists(ld_f)) ld_f <- file.path('{WORK_DIR}/ld_matrices', paste0(tag, '.ld.gz'))
  bim_f <- file.path('{WORK_DIR}/ld_matrices', paste0(tag, '.bim'))
  if (!file.exists(ld_f) || !file.exists(bim_f)) {{
    cat(sprintf('  [%d/%d] Skipping %s: LD matrix missing.\n', i, length(loci_files), tag))
    next
  }}
  gw <- fread(lf)[!duplicated(SNP)]
  bim <- fread(bim_f, header=FALSE, col.names=c('chr','SNP','cm','bp','a1_l','a2_l'))
  # Read uncompressed .ld directly or via linux gzip pipe
  R <- if (grepl('\\.gz$', ld_f)) as.matrix(fread(cmd=paste('gzip -dc', shQuote(ld_f)))) else as.matrix(fread(ld_f))
  rownames(R) <- colnames(R) <- bim$SNP
  common <- intersect(gw$SNP, bim$SNP)
  if (length(common) < 50) {{
    cat(sprintf('  [%d/%d] Skipping %s: insufficient overlap (<50 SNPs).\n', i, length(loci_files), tag))
    next
  }}
  gw <- gw[SNP %in% common]; idx <- match(gw$SNP, bim$SNP); R <- R[idx, idx, drop=FALSE]
  a1g <- toupper(gw$A1_MDD); a2g <- toupper(gw$A2_MDD); a1l <- toupper(bim$a1_l[idx]); a2l <- toupper(bim$a2_l[idx])
  flip <- (a1g == a2l) & (a2g == a1l); keep <- ((a1g == a1l) & (a2g == a2l)) | flip
  gw <- gw[keep]; R <- R[keep, keep, drop=FALSE]
  if (any(flip[keep])) gw[flip[keep], `:=`(BETA_MDD=-BETA_MDD, BETA_ANX=-BETA_ANX)]
  rownames(R) <- colnames(R) <- gw$SNP
  cat(sprintf('  [%d/%d] Fine-mapping %s (%d SNPs, N_MDD=%.0f, N_ANX=%.0f)... ', i, length(loci_files), tag, nrow(gw), median(gw$N_MDD,na.rm=TRUE), median(gw$N_ANX,na.rm=TRUE)))
  tryCatch({{
    s_mdd <- runsusie(list(beta=gw$BETA_MDD, varbeta=gw$SE_MDD^2, snp=gw$SNP, position=gw$BP, N=median(gw$N_MDD,na.rm=TRUE), type='cc', LD=R), L=10)
    s_anx <- runsusie(list(beta=gw$BETA_ANX, varbeta=gw$SE_ANX^2, snp=gw$SNP, position=gw$BP, N=median(gw$N_ANX,na.rm=TRUE), type='cc', LD=R), L=10)
    cs <- coloc.susie(s_mdd, s_anx)
    n_cs_mdd <- if (is.null(s_mdd$sets$cs)) 0 else length(s_mdd$sets$cs)
    n_cs_anx <- if (is.null(s_anx$sets$cs)) 0 else length(s_anx$sets$cs)
    top_pp <- if (!is.null(cs$summary) && nrow(cs$summary) > 0) max(cs$summary$PP.H4.abf, na.rm=TRUE) else 0
    cat(sprintf('Done! (CS_MDD=%d, CS_ANX=%d, max_PP.H4=%.3f)\n', n_cs_mdd, n_cs_anx, top_pp))
    if (!is.null(cs$summary) && nrow(cs$summary) > 0) {{
      sm <- as.data.table(cs$summary); sm[, locus := tag]; res_list[[length(res_list)+1]] <- sm
    }}
  }}, error = function(e) cat(sprintf('Failed: %s\n', e$message)))
}}
if (length(res_list) > 0) {{
  out <- rbindlist(res_list, fill=TRUE)
  out_file <- '{DRIVE_OUT}/aim2_coloc_susie_MDD_vs_ANX.tsv'
  fwrite(out, out_file, sep='\t')
  cat('\n============================================================\n')
  cat('Aim 2 coloc.susie Results (Saved to Drive: aim2_coloc_susie_MDD_vs_ANX.tsv):\n')
  cat('============================================================\n')
  print(out[order(-PP.H4.abf)][1:min(15, .N)])
}} else {{
  cat('No coloc.susie causal pairs discovered across tested loci.\n')
}}
'''
open('/content/run_aim2_susie.R', 'w').write(r_aim2)

p = subprocess.Popen(['Rscript', '/content/run_aim2_susie.R'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
p.wait()
if p.returncode != 0:
    raise subprocess.CalledProcessError(p.returncode, ['Rscript', '/content/run_aim2_susie.R'])
print('\n[SUCCESS] Aim 2 SuSiE multi-causal fine-mapping complete! Results saved in Drive.', flush=True)


In [ ]:
# [5] Aim 2 & 3 Combined: Fix ANX A1 Frequency via 1000G EUR & Run --smr-multi (Live Progress)
import os, glob, gzip, subprocess, numpy as np, pandas as pd, sys

anx_in = os.path.join(WORK_DIR, 'loci_dualtrait', 'ANX_EUR_GWAShug_aligned3.tsv.gz')
mdd_in = os.path.join(WORK_DIR, 'loci_dualtrait', 'MDD_no23andMe_EUR_SMR.ma.gz')
besd_files = sorted(glob.glob(os.path.join(DRIVE_CACHE, 'brainmeta_besd', '*.besd')))

print('[Step 5A] Aligning ANX A1 frequencies with 1000G EUR reference...', flush=True)
anx_ma_out = os.path.join(WORK_DIR, 'smr_multi', 'ANX_EUR_SMR_1000Gfreq.ma')
if os.path.exists(anx_in) and not os.path.exists(anx_ma_out):
    print('  -> Calculating 1000G EUR MAF via PLINK...', end='', flush=True)
    subprocess.run(['plink', '--bfile', os.path.join(WORK_DIR, 'g1000_eur'), '--freq', '--out', os.path.join(WORK_DIR, 'smr_multi', 'g1000_eur')], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    print(' [Done]', flush=True)
    frq = pd.read_csv(os.path.join(WORK_DIR, 'smr_multi', 'g1000_eur.frq'), sep=r'\s+', usecols=['SNP', 'A1', 'A2', 'MAF'])
    print('  -> Reading ANX GWAS summary stats...', end='', flush=True)
    anx = pd.read_csv(anx_in, sep=r'\s+', usecols=['SNP', 'A1', 'A2', 'BETA', 'SE', 'P', 'N'])
    print(f' ({len(anx):,} SNPs)', flush=True)
    m = anx.merge(frq, on='SNP', suffixes=('_gwas', '_ref')).drop_duplicates('SNP')
    a1g, a2g = m['A1_gwas'].astype(str).str.upper(), m['A2_gwas'].astype(str).str.upper()
    a1r, a2r = m['A1_ref'].astype(str).str.upper(), m['A2_ref'].astype(str).str.upper()
    m = m[((a1g == a1r) & (a2g == a2r)) | ((a1g == a2r) & (a2g == a1r))].copy()
    m['freq'] = np.where(m['A1_gwas'].astype(str).str.upper() == m['A1_ref'].astype(str).str.upper(), m['MAF'], 1.0 - m['MAF'])
    m[['SNP', 'A1_gwas', 'A2_gwas', 'freq', 'BETA', 'SE', 'P', 'N']].rename(
        columns={'A1_gwas': 'A1', 'A2_gwas': 'A2', 'BETA': 'b', 'SE': 'se', 'P': 'p', 'N': 'n'}
    ).to_csv(anx_ma_out, sep='\t', index=False)
    print(f'  -> Generated aligned ANX COJO .ma format ({len(m):,} SNPs)!', flush=True)
elif os.path.exists(anx_ma_out):
    print('  -> ANX COJO .ma already present.', flush=True)
else:
    print('  -> Note: ANX GWAS input not staged. SMR on ANX will be skipped until staged.', flush=True)

print('\n[Step 5B] Preparing MDD SMR .ma format...', flush=True)
mdd_ma_out = os.path.join(WORK_DIR, 'smr_multi', 'MDD_no23andMe_EUR_SMR.ma')
if os.path.exists(mdd_in) and not os.path.exists(mdd_ma_out):
    with gzip.open(mdd_in, 'rb') as fi, open(mdd_ma_out, 'wb') as fo:
        fo.write(fi.read())
    print('  -> MDD .ma ready for SMR.', flush=True)
elif os.path.exists(mdd_ma_out):
    print('  -> MDD .ma already prepared.', flush=True)
else:
    print('  -> Note: MDD GWAS input not staged. SMR on MDD will be skipped until staged.', flush=True)

print(f'\n[Step 5C] Checking BrainMeta BESD datasets in Drive: {len(besd_files)} found.', flush=True)
if len(besd_files) == 0:
    print(f'  [Notice] No .besd files in {DRIVE_CACHE}/brainmeta_besd yet.')
    print('  -> You can upload BrainMeta v2 Cortex eQTL BESD or custom eQTLs to this Drive folder when ready.')
else:
    for i, bf in enumerate(besd_files, 1):
        pref = bf[:-5]
        bname = os.path.basename(pref)
        for trait_tag, ma_name in [('MDD', 'MDD_no23andMe_EUR_SMR.ma'), ('ANX', 'ANX_EUR_SMR_1000Gfreq.ma')]:
            ma_path = os.path.join(WORK_DIR, 'smr_multi', ma_name)
            if os.path.exists(ma_path):
                out_p = os.path.join(DRIVE_OUT, f'{trait_tag}_{bname}_smr_multi')
                print(f'  [{i}] Running SMR-multi for {trait_tag} on {bname}...', end='', flush=True)
                subprocess.run([
                    'smr', '--bfile', os.path.join(WORK_DIR, 'g1000_eur'),
                    '--gwas-summary', ma_path, '--beqtl-summary', pref,
                    '--smr-multi', '--heidi-mtd', '1', '--peqtl-smr', '5e-8',
                    '--maf', '0.01', '--thread-num', '4', '--out', out_p
                ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                print(f' [Done -> {os.path.basename(out_p)}.msmr]', flush=True)

print('\n[SUCCESS] Pipeline completed! All output tables safely stored in:', DRIVE_OUT, flush=True)
